# Práctica · Spark DataFrames · Clientes y pedidos

Completa las celdas indicadas. Trabaja sobre el entorno dockerizado incluido en `spark_jupyter/`.

In [2]:
from iniciar_spark import get_spark
from pyspark.sql import functions as F

spark = get_spark("PracticaClientesPedidos")
print("SparkSession creada")
print("Versión de Spark:", spark.version)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/14 17:23:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession creada
Versión de Spark: 4.1.1


In [3]:
from pathlib import Path

data_dir = Path("/opt/spark-apps/datos")
clientes_path = data_dir / "clientes.csv"
pedidos_path = data_dir / "pedidos.csv"

print("Ruta clientes:", clientes_path)
print("Ruta pedidos:", pedidos_path)


Ruta clientes: /opt/spark-apps/datos/clientes.csv
Ruta pedidos: /opt/spark-apps/datos/pedidos.csv


## 1. Lee ambos CSV y muestra su esquema y algunas filas.

In [6]:
# TODO

df_clientes = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ";") \
    .csv(str(clientes_path))

df_pedidos = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ";") \
    .csv(str(pedidos_path))

# Mostrar esquemas para validar que las columnas están separadas correctamente [cite: 101, 106]
print("--- Esquema de Clientes ---")
df_clientes.printSchema()

print("--- Esquema de Pedidos ---")
df_pedidos.printSchema()

# Mostrar contenido inicial [cite: 101]
print("--- Muestra de Clientes ---")
df_clientes.show(5)

print("--- Muestra de Pedidos ---")
df_pedidos.show(5)

--- Esquema de Clientes ---
root
 |-- id_cliente: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- segmento: string (nullable = true)

--- Esquema de Pedidos ---
root
 |-- id_pedido: integer (nullable = true)
 |-- id_cliente: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- producto: string (nullable = true)
 |-- cantidad: double (nullable = true)
 |-- precio_unitario: integer (nullable = true)

--- Muestra de Clientes ---
+----------+------+----------+--------+
|id_cliente|nombre|    ciudad|segmento|
+----------+------+----------+--------+
|         1|   Ana|  Sevilla |Estandar|
|         2|  Luis|    Bilbao| Premium|
|         3| Marta| Alicante |Estandar|
|         4| Pablo|   Madrid | Premium|
|         5| Lucia|    Bilbao| Premium|
+----------+------+----------+--------+
only showing top 5 rows
--- Muestra de Pedidos ---
+---------+----------+----------+---------+--------+---------------+
|id_pedido|id_clie

## 2. Limpia el DataFrame de clientes: trim en ciudad y dropDuplicates.

In [7]:
# TODO

# Eliminar duplicados para asegurar que cada cliente sea único [cite: 12, 48, 103]
df_clientes_limpio = df_clientes.dropDuplicates()

# Aplicar trim a la columna 'ciudad' para eliminar espacios en blanco accidentales 
# También es buena práctica limpiar la columna 'nombre' si fuera necesario [cite: 104]
df_clientes_limpio = df_clientes_limpio.withColumn("ciudad", F.trim(F.col("ciudad")))

# Gestión de nulos (opcional pero recomendado según el enunciado) [cite: 14, 49, 105]
# Por ejemplo, rellenar ciudades desconocidas si las hubiera
df_clientes_limpio = df_clientes_limpio.na.fill({"ciudad": "Desconocida"})

# Mostrar resultados de la limpieza [cite: 143]
print(f"Registros originales: {df_clientes.count()}")
print(f"Registros tras eliminar duplicados: {df_clientes_limpio.count()}")

df_clientes_limpio.show(5)

Registros originales: 43
Registros tras eliminar duplicados: 40
+----------+------+--------+--------+
|id_cliente|nombre|  ciudad|segmento|
+----------+------+--------+--------+
|        18| Jorge| Sevilla| Premium|
|        40| Tomas|Valencia|Estandar|
|         5| Lucia|  Bilbao| Premium|
|         6|Carlos|Alicante|Estandar|
|        14|Sergio|  Murcia|Estandar|
+----------+------+--------+--------+
only showing top 5 rows


## 3. Convierte tipos en pedidos y crea la columna `importe`.

In [9]:
# TODO

# 3. Transformación de Pedidos

# Usamos los nombres reales: 'id_pedido', 'cantidad' y 'precio_unitario'
df_pedidos_trans = df_pedidos.withColumn("cantidad", F.col("cantidad").cast("int")) \
                             .withColumn("precio_unitario", F.col("precio_unitario").cast("double"))

# Creamos la columna 'importe'
df_pedidos_trans = df_pedidos_trans.withColumn(
    "importe", 
    F.round(F.col("cantidad") * F.col("precio_unitario"), 2)
)

# Imprimimos el esquema para estar 100% seguros de los nombres
df_pedidos_trans.printSchema()

# Mostramos los registros usando 'id_pedido' en lugar de 'pedido_id'
df_pedidos_trans.select("id_pedido", "cantidad", "precio_unitario", "importe").show(5)

root
 |-- id_pedido: integer (nullable = true)
 |-- id_cliente: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- producto: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: double (nullable = true)
 |-- importe: double (nullable = true)

+---------+--------+---------------+-------+
|id_pedido|cantidad|precio_unitario|importe|
+---------+--------+---------------+-------+
|     1001|       6|           16.0|   96.0|
|     1002|       3|           19.0|   57.0|
|     1003|       2|          210.0|  420.0|
|     1004|       4|          205.0|  820.0|
|     1005|       5|           40.0|  200.0|
+---------+--------+---------------+-------+
only showing top 5 rows


## 4. Haz un join entre clientes y pedidos.

In [11]:
# TODO

# Realizamos un inner join para obtener solo los registros con correspondencia en ambas tablas
df_join = df_clientes_limpio.join(df_pedidos_trans, "id_cliente", "inner")

# Análisis de registros
print(f"Total clientes (limpios): {df_clientes_limpio.count()}")
print(f"Total pedidos (transformados): {df_pedidos_trans.count()}")
print(f"Total registros tras el JOIN: {df_join.count()}")

# Mostrar una muestra del resultado integrado
df_join.select("id_cliente", "nombre", "id_pedido", "producto", "importe").show(10)

Total clientes (limpios): 40
Total pedidos (transformados): 120
Total registros tras el JOIN: 110
+----------+-------+---------+-----------+-------+
|id_cliente| nombre|id_pedido|   producto|importe|
+----------+-------+---------+-----------+-------+
|        12| Miguel|     1001|      Ratón|   96.0|
|        32| Andres|     1002|      Ratón|   57.0|
|         5|  Lucia|     1003|    Monitor|  420.0|
|        13|  Irene|     1004|  Impresora|  820.0|
|         5|  Lucia|     1006|    Teclado|   NULL|
|        27|  Laura|     1007|      Ratón|   23.0|
|         7|  Elena|     1008|     Webcam|  462.0|
|        18|  Jorge|     1009|    Teclado|   38.0|
|        31|Beatriz|     1010|     Webcam|   78.0|
|        36|   Gema|     1011|Auriculares|   64.0|
+----------+-------+---------+-----------+-------+
only showing top 10 rows


## 5. Filtra ventas Premium con importe >= 100.

In [12]:
# TODO

# Filtramos los datos para obtener solo aquellos cuyo importe sea mayor o igual a 100
df_premium = df_join.filter(F.col("importe") >= 100)

# Contamos cuántos registros cumplen esta condición
total_premium = df_premium.count()
print(f"Número de pedidos Premium (importe >= 100): {total_premium}")

# Mostramos los resultados ordenados por importe de forma descendente
df_premium.orderBy(F.desc("importe")).show(10)

Número de pedidos Premium (importe >= 100): 83
+----------+------+--------+--------+---------+----------+---------+--------+---------------+-------+
|id_cliente|nombre|  ciudad|segmento|id_pedido|     fecha| producto|cantidad|precio_unitario|importe|
+----------+------+--------+--------+---------+----------+---------+--------+---------------+-------+
|        22| Ruben| Granada| Premium|     1117|2025-03-16| Portátil|       6|         1031.0| 6186.0|
|         6|Carlos|Alicante|Estandar|     1061|2025-03-11| Portátil|       4|         1076.0| 4304.0|
|        37| Oscar|Alicante| Premium|     1013|2025-02-16| Portátil|       5|          741.0| 3705.0|
|        10| David|  Bilbao|Estandar|     1091|2025-03-11| Portátil|       3|         1139.0| 3417.0|
|         2|  Luis|  Bilbao| Premium|     1092|2025-02-09| Portátil|       3|         1104.0| 3312.0|
|         3| Marta|Alicante|Estandar|     1075|2025-02-21| Portátil|       3|          883.0| 2649.0|
|        10| David|  Bilbao|Estanda

## 6. Clasifica pedidos con `when` en Alto / Medio / Bajo.

In [13]:
# TODO

# Definimos los rangos de clasificación:
# - Alto: >= 200
# - Medio: >= 100 y < 200
# - Bajo: < 100 (el resto)

df_clasificado = df_join.withColumn(
    "nivel_pedido",
    F.when(F.col("importe") >= 200, "Alto")
     .when((F.col("importe") >= 100) & (F.col("importe") < 200), "Medio")
     .otherwise("Bajo")
)

# Seleccionamos columnas clave para verificar que la lógica se aplicó correctamente
print("Muestra de pedidos clasificados:")
df_clasificado.select("id_pedido", "importe", "nivel_pedido").show(10)

Muestra de pedidos clasificados:
+---------+-------+------------+
|id_pedido|importe|nivel_pedido|
+---------+-------+------------+
|     1088|  124.0|       Medio|
|     1024|  252.0|        Alto|
|     1009|   38.0|        Bajo|
|     1094|  768.0|        Alto|
|     1056|  108.0|       Medio|
|     1112|  176.0|       Medio|
|     1045|  105.0|       Medio|
|     1006|   NULL|        Bajo|
|     1003|  420.0|        Alto|
|     1119|  444.0|        Alto|
+---------+-------+------------+
only showing top 10 rows


## 7. Calcula agregaciones por ciudad y segmento.

In [14]:
# TODO

# Agrupamos por ciudad y el nivel de pedido creado en el paso anterior
# Calculamos: número de pedidos, suma total de ingresos y el importe medio
df_metricas = df_clasificado.groupBy("ciudad", "nivel_pedido").agg(
    F.count("id_pedido").alias("num_pedidos"),
    F.sum("importe").alias("ingresos_totales"),
    F.round(F.avg("importe"), 2).alias("promedio_importe")
)

# Ordenamos los resultados por ciudad para que sea más legible
df_metricas_final = df_metricas.orderBy("ciudad", F.desc("ingresos_totales"))

# Mostrar el resultado de las agregaciones
print("Métricas de ventas por ciudad y nivel de pedido:")
df_metricas_final.show()

Métricas de ventas por ciudad y nivel de pedido:
+--------+------------+-----------+----------------+----------------+
|  ciudad|nivel_pedido|num_pedidos|ingresos_totales|promedio_importe|
+--------+------------+-----------+----------------+----------------+
|Alicante|        Alto|         11|         15855.0|         1441.36|
|Alicante|       Medio|          3|           484.0|          161.33|
|Alicante|        Bajo|          3|           246.0|            82.0|
|  Bilbao|        Alto|         18|         16578.0|           921.0|
|  Bilbao|       Medio|          4|           576.0|           144.0|
|  Bilbao|        Bajo|          5|           205.0|           51.25|
| Granada|        Alto|          8|          9745.0|         1218.13|
| Granada|       Medio|          3|           466.0|          155.33|
| Granada|        Bajo|          1|            78.0|            78.0|
|  Madrid|        Alto|          6|          4528.0|          754.67|
|  Madrid|       Medio|          3|      

## 8. Crea una vista temporal y haz una consulta SQL.

In [ ]:
# TODO

## 9. Usa `sample()` y `randomSplit()`.

In [ ]:
# TODO

## 10. Guarda el resultado en Parquet en `/opt/spark-apps/salida/resultado_parquet` y léelo de nuevo.

In [ ]:
# TODO

## 11. Responde brevemente en Markdown:
- ¿Qué ventaja tiene usar join?
- ¿Qué diferencia hay entre sample y randomSplit?
- ¿Qué pedido se pierde en el inner join y por qué?

## 12. Cierra la sesión de Spark.

In [ ]:
spark.stop()